## Spending Analyser Built from Python,NumPy, Pandas
Built by - Dibya

In [187]:
# Importing Libraries
import numpy as np
import pandas as pd

In [188]:
df = pd.read_csv('rahul.csv')


## Date Parser

    It parses the dates in different formats and converts it to YYYY-MM-DD format

In [189]:
# Feature 1
# Date Parser built on pandas
def parse_date(date_s):
    if pd.isna(date_s):
        return pd.NaT
    s = str(date_s).strip()
    if s == '' or s.lower() in ('nan', 'none', 'nat'):
        return pd.NaT
    formats = ['%d/%m/%y', '%d/%m/%Y', '%Y-%m-%d', '%Y/%m/%d', '%d-%b-%y', '%d-%b-%Y',
               '%d %b %Y', '%d %B %Y', '%Y-%m-%d %H:%M:%S']
    for fmt in formats:
        try:
            return pd.to_datetime(s, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.to_datetime(s, errors='coerce', dayfirst=True)

df['Date'] = df['Date'].apply(parse_date)

# Cleaning 'Amount' column
df['Amount'] = (
    df['Amount'].astype(str)
    .str.replace('₹', '', regex=False)
    .str.replace('Rs.', '', regex=False)
    .str.replace('Rs', '', regex=False)
    .str.replace(',', '', regex=False)
    .str.strip()
)
df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')

# Standarding Type  to include debit or credit
df['Type'] = df['Type'].str.lower().str.strip()
df['Type'] = df['Type'].replace({'dr': 'debit', 'cr': 'credit'})

# Cleaning Duplicates
before = len(df)
df = df.drop_duplicates(subset=['Date', 'Time', 'Description', 'Type', 'Amount', 'Balance', 'Mode'], keep='first')
dropped = before - len(df)

# parser outcome
unparseable_amounts = df['Amount'].isna().sum()
unparseable_dates = df['Date'].isna().sum()
months_covered = df['Date'].dt.to_period('M').nunique()
print(f"Parsed {len(df)} transactions across {months_covered} months. "
      f"Dropped {dropped} duplicates. "
      f"{unparseable_amounts} unparseable amounts, {unparseable_dates} unparseable dates.")

Parsed 1310 transactions across 6 months. Dropped 18 duplicates. 0 unparseable amounts, 0 unparseable dates.


## Clean Vendor names

It Cleans the vendor datas and gives the actual vendor name

In [190]:
# Feature 2
# Vendor keywords dictionary
vendor_keywords = {
    'Swiggy': ['swiggy'],
    'Swiggy Instamart': ['instamart', 'bundl'],
    'Zomato': ['zomato'],
    'Zomato Dining': ['zomato-dining'],
    'Zepto': ['zepto', 'kirana'],
    'Blinkit': ['blinkit'],
    'BigBasket': ['bigbasket'],
    'Grofers': ['grofers'],
    'Dmart': ['dmart', 'avenue supermarts', 'innovative retail'],
    'Restaurant': ['restaurant', 'dineout', 'meghana', 'truffles'],
    'Amazon': ['amazon', 'amzn'],
    'Amazon Prime': ['amazon prime', 'amzn prime'],
    'Nykaa': ['nykaa', 'fsn e-commerce'],
    'Flipkart': ['flipkart', 'fkart'],
    'Myntra': ['myntra'],
    'Uber': ['uber'],
    'Ola': ['ola', 'ani technologies'],
    'Rapido': ['rapido','roppen'],
    'BMTC': ['bmtc', 'tummoc'],
    'Fuel/Petrol': ['petrol', 'bpcl', 'indian oil', 'ioc'],
    'Starbucks': ['starbucks'],
    'Third Wave Coffee': ['thirdwave', 'third wave', 'twc india'],
    'Cafe Coffee Day': ['coffee day', 'ccd'],
    'Netflix': ['netflix'],
    'Spotify': ['spotify'],
    'Disney+ Hotstar': ['hotstar', 'star india'],
    'Zerodha': ['zerodha'],
    'Groww': ['groww'],
    'Cash Withdrawal': ['atm-wdl'],
    'Personal Transfer': ['upi-aman-', 'upi-ankit-', 'upi-priya-', 'upi-neha-', 'upi-vikas-', 'upi-karan-', 'upi-sneha-'],
    'Rent': ['rent-landlord'],
    'Salary': ['salary'],
    'BookMyShow': ['bookmyshow', 'bigtree', 'bms movie'],
    'Airtel': ['airtel'],
    'Vi': ['vodafone', 'vi postpaid', 'vi-recharge'],
    'Jio': ['jio'],
    'BESCOM': ['bescom', 'bangalore elec'],
    'BWSSB': ['bwssb'],
}
df['desc_lower'] = df['Description'].str.lower()

# Finds the vendor from Description of Transections
def clean_vendor(desc_lower):
    if any(k in desc_lower for k in vendor_keywords['Zomato Dining']):
        return 'Zomato Dining'
    for clean_name, keywords in vendor_keywords.items():
        if clean_name == 'Zomato Dining':
            continue
        if any(k in desc_lower for k in keywords):
            return clean_name
    return desc_lower

df['vendor_clean'] = df['desc_lower'].apply(clean_vendor)


# Printing Vendor names

print(df['vendor_clean'].value_counts().head(10))



vendor_clean
Swiggy              196
Zomato              101
Ola                  87
Amazon               86
Restaurant           73
Uber                 71
Zepto                71
Rapido               55
Flipkart             47
Swiggy Instamart     47
Name: count, dtype: int64


## Vendor Categorization

it groups vendors categories

In [191]:
# Feature 3
# Category mapping dictionary
category_mapping = {
    'Swiggy': 'Food Delivery', 'Zomato': 'Food Delivery',
    'Zomato Dining': 'Restaurants', 'Restaurant': 'Restaurants',
    'Swiggy Instamart': 'Quick Commerce', 'Zepto': 'Quick Commerce',
    'Blinkit': 'Quick Commerce', 'BigBasket': 'Quick Commerce', 'Grofers': 'Quick Commerce',
    'Amazon': 'Ecommerce', 'Flipkart': 'Ecommerce', 'Myntra': 'Ecommerce', 'Nykaa': 'Ecommerce',
    'Uber': 'Transport', 'Ola': 'Transport', 'Rapido': 'Transport', 'BMTC': 'Transport',
    'Fuel/Petrol': 'Fuel',
    'Starbucks': 'Cafe', 'Third Wave Coffee': 'Cafe', 'Cafe Coffee Day': 'Cafe',
    'Netflix': 'Subscriptions', 'Spotify': 'Subscriptions', 'Disney+ Hotstar': 'Subscriptions',
    'Dmart': 'Groceries',
    'Zerodha': 'Investments', 'Groww': 'Investments',
    'Cash Withdrawal': 'Cash Withdrawal',
    'Personal Transfer': 'Personal Transfer',
    'Rent': 'Rent',
    'Salary': 'Salary',
    'Amazon Prime': 'Subscriptions',
    'BookMyShow': 'Entertainment',
    'Airtel': 'Subscriptions', 'Vi': 'Subscriptions', 'Jio': 'Subscriptions',
    'BESCOM': 'Utilities', 'BWSSB': 'Utilities',
}

# If transection not classified save it to Uncategorised
df['Category'] = df['vendor_clean'].map(category_mapping).fillna('Uncategorised')
analysis_df = df[(df['Type'] == 'debit') &
                  (~df['Category'].isin(['Personal Transfer', 'Cash Withdrawal', 'Uncategorised']))].copy()
print(df['Category'].value_counts())

Category
Food Delivery        297
Transport            250
Quick Commerce       184
Ecommerce            172
Cafe                  99
Restaurants           93
Subscriptions         51
Groceries             30
Fuel                  28
Utilities             23
Investments           23
Personal Transfer     18
Cash Withdrawal       17
Entertainment         13
Salary                 6
Rent                   6
Name: count, dtype: int64


##  Spending Overview
shows the cash flow of Rahul

In [192]:
# Feature 4
# ── Spending Overview ──────────────────────────────────────

# 1. Total credits and debits (use full df — includes salary, transfers, everything)
total_credits = df[df['Type'] == 'credit']['Amount'].sum()
total_debits = df[df['Type'] == 'debit']['Amount'].sum()

# 2. Net savings and savings rate
net_savings = total_credits - total_debits
savings_rate = (net_savings / total_credits) * 100

# 3. Top 5 categories by spend (use analysis_df — excludes transfers/withdrawals/uncategorised)
top_categories = (analysis_df.groupby('Category')['Amount']
                   .sum()
                   .sort_values(ascending=False)
                   .head(5))

# 4. Top 5 vendors by spend
top_vendors = (analysis_df.groupby('vendor_clean')['Amount']
               .sum()
               .sort_values(ascending=False)
               .head(5))

# 5. Total transaction count
total_transactions = len(df)

# ── Print the executive summary ────────────────────────────
print("=" * 50)
print("SPENDING OVERVIEW")
print("=" * 50)
print(f"Total Credits:      ₹{total_credits:,.0f}")
print(f"Total Debits:       ₹{total_debits:,.0f}")
print(f"Net Savings:        ₹{net_savings:,.0f}")
print(f"Savings Rate:       {savings_rate:.1f}%")
print(f"Total Transactions: {total_transactions}")
print()
print("Top 5 Categories by Spend:")
for cat, amt in top_categories.items():
    print(f"  {cat:<20} ₹{amt:,.0f}")
print()
print("Top 5 Vendors by Spend:")
for vendor, amt in top_vendors.items():
    print(f"  {vendor:<20} ₹{amt:,.0f}")

SPENDING OVERVIEW
Total Credits:      ₹509,774
Total Debits:       ₹1,678,901
Net Savings:        ₹-1,169,127
Savings Rate:       -229.3%
Total Transactions: 1310

Top 5 Categories by Spend:
  Ecommerce            ₹603,877
  Investments          ₹248,160
  Food Delivery        ₹129,470
  Restaurants          ₹127,290
  Rent                 ₹108,000

Top 5 Vendors by Spend:
  Amazon               ₹328,530
  Zerodha              ₹210,000
  Flipkart             ₹177,510
  Restaurant           ₹117,737
  Rent                 ₹108,000


## Monthly Trend
Gives  spending beheviour of a preson in a month

In [193]:
# Feature 5
# ── Monthly Trend Analysis ─────────────────────────────────
import numpy as np

# Ensure Date is datetime
analysis_df['Date'] = pd.to_datetime(analysis_df['Date'])

# Extract month name
analysis_df['month'] = analysis_df['Date'].dt.strftime('%b')

month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']

# Build the category x month matrix
month_pivot = analysis_df.pivot_table(
    values='Amount',
    index='Category',
    columns='month',
    aggfunc='sum',
    fill_value=0
)

# Reorder columns chronologically
month_pivot = month_pivot.reindex(columns=[m for m in month_order if m in month_pivot.columns])


#  Compute month-on-month growth (first month -> last month)
first_month = month_pivot.columns[0]
last_month = month_pivot.columns[-1]

# Avoid division by zero — use NaN for categories with 0 in first month
growth_pct = np.where(
    month_pivot[first_month] == 0,
    np.nan,
    (month_pivot[last_month] - month_pivot[first_month]) / month_pivot[first_month] * 100
)

growth = pd.Series(growth_pct, index=month_pivot.index).sort_values(ascending=False)

biggest_growth_cat = growth.idxmax()
biggest_decline_cat = growth.idxmin()

print(f"Trending UP:   {biggest_growth_cat} ({growth[biggest_growth_cat]:+.1f}% from {first_month} to {last_month})")
print(f"Trending DOWN: {biggest_decline_cat} ({growth[biggest_decline_cat]:+.1f}% from {first_month} to {last_month})")
print()
print("Full growth ranking (first month → last month):")
print(growth.round(1))

Trending UP:   Cafe (+57.2% from Jan to Jun)
Trending DOWN: Fuel (-90.5% from Jan to Jun)

Full growth ranking (first month → last month):
Category
Cafe              57.2
Entertainment     51.5
Ecommerce         40.0
Restaurants       32.2
Utilities         22.9
Food Delivery      6.1
Subscriptions      5.8
Rent               0.0
Quick Commerce   -10.3
Transport        -36.4
Investments      -39.3
Groceries        -67.3
Fuel             -90.5
dtype: float64


## Spending Pattern
it tells about the spending habits and shows activity of the person

In [194]:
# Feature 6
# ── Time-of-Day  Analysis ───
import pandas as pd
import numpy as np

analysis_df['hour'] = analysis_df['Time'].str[:2].astype(int)

def time_bucket(hour):
    if 5 <= hour < 12: return 'Morning'
    elif 12 <= hour < 18: return 'Afternoon'
    elif 18 <= hour < 24: return 'Evening'
    else: return 'Late Night'

analysis_df['time_bucket'] = analysis_df['hour'].apply(time_bucket)
bucket_order = ['Morning', 'Afternoon', 'Evening', 'Late Night']

bucket_counts = analysis_df.pivot_table(
    values='Amount', index='Category', columns='time_bucket',
    aggfunc='count', fill_value=0
).reindex(columns=bucket_order, fill_value=0)

bucket_pct = bucket_counts.div(bucket_counts.sum(axis=1), axis=0) * 100
bucket_pct = bucket_pct.round(1)

# Building Heatmap
W = 10

def shade(pct):
    if pct == 0: symbol = "."
    elif pct <= 15: symbol = "░"
    elif pct <= 35: symbol = "▒"
    elif pct <= 60: symbol = "▓"
    else: symbol = "█"
    return f"{symbol} {pct:5.1f}%".ljust(W)

print("=" * 80)
print("SPENDING HEATMAP — Category x Time of Day (% of category total)")
print("=" * 80)
print()

name_w = max(len(c) for c in bucket_pct.index) + 2
header = " " * name_w + "".join(b.ljust(W) for b in bucket_order)
print(header)

for cat in bucket_pct.index:
    row = bucket_pct.loc[cat]
    line = f"{cat:<{name_w}}"
    for b in bucket_order:
        line += shade(row[b])
    print(line)

print()
print("Legend: .  0%   ░ ≤15%   ▒ ≤35%   ▓ ≤60%   █ >60%")
print()

# ── Insights (top 4 patterns) ────────────────────
print("=" * 80)
print("KEY INSIGHTS")
print("=" * 80)
print()

insight_candidates = []
for cat in bucket_pct.index:
    row = bucket_pct.loc[cat]
    if row.sum() == 0:
        continue
    top_bucket = row.idxmax()
    insight_candidates.append((row[top_bucket], cat, top_bucket))

# Sort by strongest concentration
insight_candidates.sort(reverse=True)
for pct, cat, bucket in insight_candidates[:4]:
    print(f"• {pct:.0f}% of {cat} transactions happen in the {bucket}")

print()
print("Overall spend timing (all categories combined, % of total transactions):")
overall_counts = bucket_counts.sum(axis=0)
overall_pct = (overall_counts / overall_counts.sum() * 100).round(1)
for b in bucket_order:
    print(f"  {b:<12} {overall_pct[b]:5.1f}%")

SPENDING HEATMAP — Category x Time of Day (% of category total)

                Morning   Afternoon Evening   Late Night
Cafe            ▓  38.4%  ▓  38.4%  ▒  16.2%  ░   7.1%  
Ecommerce       ▒  30.2%  ▒  23.8%  ▒  25.0%  ▒  20.9%  
Entertainment   ▒  23.1%  ▒  15.4%  ▓  46.2%  ▒  15.4%  
Food Delivery   ▒  19.2%  ▒  26.6%  ▓  47.1%  ░   7.1%  
Fuel            ▒  21.4%  ▓  35.7%  ▒  25.0%  ▒  17.9%  
Groceries       ▒  30.0%  ░  13.3%  ▒  20.0%  ▓  36.7%  
Investments     ▓  52.2%  ░   4.3%  ▒  26.1%  ▒  17.4%  
Quick Commerce  ▒  21.2%  ▒  27.2%  ▓  44.6%  ░   7.1%  
Rent            ▒  33.3%  █  66.7%  .   0.0%  .   0.0%  
Restaurants     ▒  15.1%  ▒  29.0%  ▓  51.6%  ░   4.3%  
Subscriptions   ▒  27.5%  ▒  19.6%  ░  13.7%  ▓  39.2%  
Transport       ▒  34.0%  ▒  32.8%  ▒  28.8%  ░   4.4%  
Utilities       ▒  26.1%  ▒  34.8%  ▒  17.4%  ▒  21.7%  

Legend: .  0%   ░ ≤15%   ▒ ≤35%   ▓ ≤60%   █ >60%

KEY INSIGHTS

• 67% of Rent transactions happen in the Afternoon
• 52% of Investments

## Anomaly Detection
Shows anomaly in spending and where it going

In [195]:
# Feature 7
# ── Anomaly Detection ──────────────────────────────────────

# Compute per-category mean and std, broadcast back to each row via transform
analysis_df['category_mean'] = analysis_df.groupby('Category')['Amount'].transform('mean')
analysis_df['category_std'] = analysis_df.groupby('Category')['Amount'].transform('std')

# Z-score for each transaction relative to its own category
analysis_df['z_score'] = (analysis_df['Amount'] - analysis_df['category_mean']) / analysis_df['category_std']

# Flag anomalies: z_score > 2 (top ~2% of category, unusually large spend)
anomalies = analysis_df[analysis_df['z_score'] > 2].sort_values('z_score', ascending=False)

print(f"Total anomalies detected: {len(anomalies)}")
print()
print("Top 5 Anomalies:")
print("=" * 80)
top5 = anomalies[['Date', 'vendor_clean', 'Category', 'Amount', 'z_score']].head(5)
for _, row in top5.iterrows():
    print(f"{row['Date'].strftime('%Y-%m-%d') if hasattr(row['Date'], 'strftime') else row['Date']}  "
          f"{row['vendor_clean']:<20} {row['Category']:<15} ₹{row['Amount']:>8,.0f}  z={row['z_score']:.2f}")

print()
print("Anomaly breakdown by category:")
print(anomalies['Category'].value_counts())


Total anomalies detected: 33

Top 5 Anomalies:
2024-03-12  BigBasket            Quick Commerce  ₹   1,865  z=4.85
2024-01-23  BigBasket            Quick Commerce  ₹   1,797  z=4.60
2024-02-26  Restaurant           Restaurants     ₹   8,383  z=4.35
2024-06-26  Amazon               Ecommerce       ₹  22,008  z=4.09
2024-02-07  Amazon               Ecommerce       ₹  21,986  z=4.09

Anomaly breakdown by category:
Category
Ecommerce         15
Quick Commerce     6
Restaurants        5
Food Delivery      3
Subscriptions      2
Fuel               2
Name: count, dtype: int64


## Spending Archetype
a persons habit and prirorities are exposes by this

In [196]:
# feature 8
# ── Spending Archetype Detection ───────────────────────────

# matching "X% of debits" in the brief's wording
category_spend = analysis_df.groupby('Category')['Amount'].sum()

def pct_of_debits(amount):
    return (amount / total_debits) * 100

archetypes_matched = []

# 1. THE FOODIE — Food Delivery + Restaurants + Cafe > 25% of debits
def check_foodie():
    spend = category_spend.get('Food Delivery', 0) + category_spend.get('Restaurants', 0) + category_spend.get('Cafe', 0)
    pct = pct_of_debits(spend)
    if pct > 25:
        return ("The Foodie", f"Food Delivery + Restaurants + Cafe = {pct:.0f}% of debits")
    return None

# 2. THE QUICK COMMERCE JUNKIE — Quick Commerce > 15% of debits
def check_quick_commerce_junkie():
    pct = pct_of_debits(category_spend.get('Quick Commerce', 0))
    if pct > 15:
        return ("The Quick Commerce Junkie", f"Quick Commerce = {pct:.0f}% of debits")
    return None

# 3. THE SHOPAHOLIC — Ecommerce > 15% of debits
def check_shopaholic():
    pct = pct_of_debits(category_spend.get('Ecommerce', 0))
    if pct > 15:
        return ("The Shopaholic", f"Ecommerce = {pct:.0f}% of debits")
    return None

# 4. THE INVESTOR — Investments > 15% of debits
def check_investor():
    pct = pct_of_debits(category_spend.get('Investments', 0))
    if pct > 15:
        return ("The Investor", f"Investments = {pct:.0f}% of debits")
    return None

# Late-night window: 21:00-01:00 (9 PM to 2 AM)
def check_late_night_snacker():
    fd = analysis_df[analysis_df['Category'] == 'Food Delivery']
    if len(fd) == 0:
        return None
    late_night = fd[(fd['hour'] >= 21) | (fd['hour'] <= 2)]
    pct = (len(late_night) / len(fd)) * 100
    if pct > 50:
        return ("The Late-Night Snacker", f"{pct:.0f}% of Food Delivery orders happen between 21:00-01:00")
    return None

# 6. THE CAB COMMUTER — Transport > 10% of debits
def check_cab_commuter():
    pct = pct_of_debits(category_spend.get('Transport', 0))
    if pct > 10:
        return ("The Cab Commuter", f"Transport = {pct:.0f}% of debits")
    return None

# 7. THE SUBSCRIPTION LOVER — 5+ distinct subscription vendors active
def check_subscription_lover():
    subs = analysis_df[analysis_df['Category'] == 'Subscriptions']['vendor_clean'].nunique()
    if subs >= 5:
        return ("The Subscription Lover", f"{subs} distinct subscription vendors active")
    return None

# 8. THE YOLO SPENDER — savings rate below 10%
def check_yolo_spender():
    if savings_rate < 10:
        return ("The YOLO Spender", f"Savings rate = {savings_rate:.1f}%")
    return None

# 9. THE DISCIPLINED SAVER — savings rate above 40%
def check_disciplined_saver():
    if savings_rate > 40:
        return ("The Disciplined Saver", f"Savings rate = {savings_rate:.1f}%")
    return None

# 10. BONUS — invented archetype: THE PAVEMENT COFFEE CONNOISSEUR
# Rule: Cafe spend > 8% of debits AND cafe transaction count > 15
# (captures frequent small-ticket coffee habit, distinct from occasional big-ticket cafe visits)
def check_pavement_coffee_connoisseur():
    cafe_df = analysis_df[analysis_df['Category'] == 'Cafe']
    cafe_pct = pct_of_debits(category_spend.get('Cafe', 0))
    cafe_count = len(cafe_df)
    if cafe_pct > 8 and cafe_count > 15:
        return ("The Pavement Coffee Connoisseur",
                f"Cafe = {cafe_pct:.1f}% of debits across {cafe_count} transactions (frequent small-ticket habit)")
    return None

# ── Run all checks ──────────────────────────────────────────
archetypes_matched = []   # ← reset here, every time this cell runs

checks = [
    check_foodie, check_quick_commerce_junkie, check_shopaholic, check_investor,
    check_late_night_snacker, check_cab_commuter, check_subscription_lover,
    check_yolo_spender, check_disciplined_saver, check_pavement_coffee_connoisseur
]

print("=" * 60)
print("SPENDING ARCHETYPES MATCHED")
print("=" * 60)
for check_fn in checks:
    result = check_fn()
    if result:
        name, metric = result
        archetypes_matched.append(name)
        print(f"🏷️  {name:<32} — {metric}")

print()
print(f"Total archetypes matched: {len(archetypes_matched)}")
print("Summary: " + " + ".join(archetypes_matched))


SPENDING ARCHETYPES MATCHED
🏷️  The Shopaholic                   — Ecommerce = 36% of debits
🏷️  The Subscription Lover           — 6 distinct subscription vendors active
🏷️  The YOLO Spender                 — Savings rate = -229.3%

Total archetypes matched: 3
Summary: The Shopaholic + The Subscription Lover + The YOLO Spender


In [197]:
import numpy as np

# Fix late-night window to match spec exactly: 21:00-01:00 (not 02:00)
def check_late_night_snacker():
    fd = analysis_df[analysis_df['Category'] == 'Food Delivery']
    if len(fd) == 0:
        return None
    late_night = fd[(fd['hour'] >= 21) | (fd['hour'] <= 1)]
    pct = (len(late_night) / len(fd)) * 100
    if pct > 50:
        return ("The Late-Night Snacker", f"{pct:.0f}% food after 9 PM", pct)
    return None

start_date = df['Date'].min().strftime('%b %Y')
end_date = df['Date'].max().strftime('%b %Y')
unique_vendors = df['vendor_clean'].nunique()

print("=" * 65)
print(f"  SpendDNA REPORT  -  RAHUL SHARMA")
print(f"  6 months  -  {len(df)} transactions  -  {start_date} to {end_date}")
print("=" * 65)

print()
print("  EXECUTIVE SUMMARY")
print(f"    Total credits      : Rs. {total_credits:,.0f}")
print(f"    Total debits       : Rs. {total_debits:,.0f}")
print(f"    Net change         : -Rs. {abs(net_savings):,.0f}  ({'overspending' if net_savings < 0 else 'saving'})")
print(f"    Savings rate       : {savings_rate:.1f}%")
print(f"    Transactions       : {len(df)}")
print(f"    Unique vendors     : {unique_vendors}")

print()
print("  TOP CATEGORIES (% of debit total)")
# Calculate percentages and store them to find the maximum
category_percentages = {}
for cat, amt in top_categories.items():
    category_percentages[cat] = pct_of_debits(amt)

# Find the maximum percentage to normalize bar lengths
if category_percentages:
    max_pct_for_bars = max(category_percentages.values())
else:
    max_pct_for_bars = 1 # Default to 1 to avoid division by zero if no categories

MAX_BAR_LENGTH = 20 # Define a consistent max bar length

for cat, amt in top_categories.items():
    pct = category_percentages[cat]
    # Calculate scaled bar length
    if max_pct_for_bars > 0:
        bar_len = int((pct / max_pct_for_bars) * MAX_BAR_LENGTH)
    else:
        bar_len = 0 # If max_pct is 0, all lengths are 0
    bar = '#' * bar_len
    print(f"    {cat:<16} {bar:<{MAX_BAR_LENGTH}} {pct:>5.1f}%   Rs. {amt:>10,.0f}")

print()
print("  TOP VENDORS")
vendor_orders = analysis_df.groupby('vendor_clean').size()
top_vendors = top_vendors.sort_values(ascending=False)
for vendor, amt in top_vendors.items():
    print(f"    {vendor:<16} Rs. {amt:>10,.0f}  ({vendor_orders[vendor]} orders)")

print()
print("  TIME-OF-DAY PATTERNS")
fd_late = check_late_night_snacker()
print(f"    Food Delivery peaks: 21:00 - 01:00  ({fd_late[2]:.0f}% of orders)" if fd_late else "    Food Delivery: no strong peak")

print()
print("  MONTHLY TREND (Food Delivery)")
if 'Food Delivery' in month_pivot.index:
    fd_trend = month_pivot.loc['Food Delivery']
    max_val = fd_trend.max()
    for month, amt in fd_trend.items():
        bar_len = int((amt / max_val) * 20) if max_val > 0 else 0
        print(f"    {month}  Rs. {amt:>7,.0f}  {'#' * int(bar_len/2)}")

print()
print("  TOP 5 ANOMALIES")
if len(anomalies) > 0:
    top_5 = anomalies.head(5)
    for _, row in top_5.iterrows():
        print(f"    {row['Date'].strftime('%d %b')} - {row['vendor_clean']:<12} Rs. {row['Amount']:>6,.0f}  (z={row['z_score']:.1f})")
else:
    print("    No anomalies found.")

print()
print("  RAHUL'S SPENDING ARCHETYPES")
for check_fn in [check_foodie, check_quick_commerce_junkie, check_shopaholic,
                  check_investor, check_late_night_snacker, check_cab_commuter,
                  check_subscription_lover, check_yolo_spender, check_disciplined_saver,
                  check_pavement_coffee_connoisseur]:
    result = check_fn()
    if result:
        name, metric = result[0], result[1]
        print(f"    -> {name:<24} ({metric})")

print()
print("=" * 65)
print("  KEY INSIGHTS")
food_delivery_pct = pct_of_debits(category_spend.get('Food Delivery', 0))
qc_ecom_food_pct = pct_of_debits(category_spend.get('Food Delivery', 0) +
                                   category_spend.get('Quick Commerce', 0) +
                                   category_spend.get('Ecommerce', 0))
monthly_burn = abs(net_savings) / 6
burn_pct_of_salary = (monthly_burn / (total_credits / 6)) * 100

print(f"  -> Rahul is burning through his savings at Rs. {monthly_burn:,.0f} per month")
if burn_pct_of_salary > 0:
    print(f"     - {burn_pct_of_salary:.0f}% over his salary.")
if fd_late:
    print(f"  -> {fd_late[2]:.0f}% of his food-delivery spend happens after 9 PM")
    print(f"     suggesting stress eating or single-living patterns.")
print(f"  -> Investments ({pct_of_debits(category_spend.get('Investments', 0)):.1f}%) are healthy, but discretionary")
print(f"     (Food + Q-com + E-com = {qc_ecom_food_pct:.0f}%) dominates his wallet.")
print("=" * 65)


  SpendDNA REPORT  -  RAHUL SHARMA
  6 months  -  1310 transactions  -  Jan 2024 to Jun 2024

  EXECUTIVE SUMMARY
    Total credits      : Rs. 509,774
    Total debits       : Rs. 1,678,901
    Net change         : -Rs. 1,169,127  (overspending)
    Savings rate       : -229.3%
    Transactions       : 1310
    Unique vendors     : 37

  TOP CATEGORIES (% of debit total)
    Ecommerce        ####################  36.0%   Rs.    603,877
    Investments      ########              14.8%   Rs.    248,160
    Food Delivery    ####                   7.7%   Rs.    129,470
    Restaurants      ####                   7.6%   Rs.    127,290
    Rent             ###                    6.4%   Rs.    108,000

  TOP VENDORS
    Amazon           Rs.    328,530  (86 orders)
    Zerodha          Rs.    210,000  (14 orders)
    Flipkart         Rs.    177,510  (47 orders)
    Restaurant       Rs.    117,737  (73 orders)
    Rent             Rs.    108,000  (6 orders)

  TIME-OF-DAY PATTERNS
    Food Deli

## AI Assistance Disclosure
This project was implemented by me using Python, Pandas, and NumPy.

I used ChatGPT only for:

->Syntax guidance

->Debugging

->Code review

All outputs, spending percentages, anomaly detection results, archetype detection, and final conclusions were generated by running the code on the provided dataset.